# Gas Transfer Visualization Demo

This notebook demonstrates how to visualize gas transfer and control parameters in AnimaKinetica.

## Features:
- Plot dissolved oxygen (DO) concentration
- Visualize stirring speed control (rpm)
- Monitor gas flow rate (vvm)
- Track oxygen mole fraction in gas phase
- Analyze cumulative gas consumption across batches

---

In [32]:
# Import necessary modules
import numpy as np
from animakinetica.core.parameters import SimulationParameters
from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.analysis.gas_visualization import (
    plot_batch_gas_transfer,
    plot_simulation_cumulative_gas_use
)
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy

## 1. Initialize Simulation

Load parameters and create a simulation instance.

In [33]:
# Load default parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "excess_kinetics_batch","fed_batch"])
# Create simulation with EXCESS kinetics and FEDBATCH mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.EXCESS,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    operating_mode=OperatingMode.FEDBATCH
)

print(f"Simulation Type: {sim.data.simulation_type}")

Loading parameters from 6 file(s) sequentially...
  [1/6] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\excess_kinetics\excess_kinetics_batch.yaml
  [6/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakineti

## 2. Run Single Batch Operation

Execute a single batch to collect gas transfer data.

In [34]:
# Run single batch
results = sim.run_simulation()
batch1 = sim.data.batches[0]

print(f"\nBatch completed in {batch1.t_end:.1f} hours")
print(f"Final cell density: {batch1.Xv_final/1e9:.2f} x 10^6 cells/mL")
print(f"Final volume: {batch1.V_final:.2f} L")

run_simulation execution time: 116.992 ms

Batch completed in 100.0 hours
Final cell density: 3.08 x 10^6 cells/mL
Final volume: 1600.00 L


## 3. Plot Gas Transfer Parameters

Visualize all gas transfer control parameters in a 4-panel figure.

In [35]:
# Plot gas transfer parameters for batch 1
fig_gas = plot_batch_gas_transfer(sim, batch_number=1)
fig_gas.show()

# Alternative: using the data object method
# fig_gas = sim.data.batches[0].plot_gas_transfer()
# fig_gas.show()

### Interpretation:

**Panel 1 - Dissolved Oxygen (DO):**
- Target: Maintained at setpoint (typically ~0.2 mmol/L)
- PID controller adjusts stirring and gas composition to maintain DO
- Deviations indicate controller response to changing oxygen demand

**Panel 2 - Stirring Speed (rpm):**
- Primary control variable for oxygen transfer
- Increases as cell density and O2 demand increase
- Limited by maximum rpm to prevent cell damage

**Panel 3 - Gas Flow Rate (vvm):**
- Secondary control variable
- Volume per volume per minute (vvm = L_gas/L_reactor/min)
- Adjusted when stirring reaches limits

**Panel 4 - Oxygen Mole Fraction:**
- Fraction of O2 in gas phase (pure air ≈ 0.21)
- Can be enriched (>0.21) when other controls are insufficient
- Cost increases with O2 enrichment

## 4. Analyze Control Strategy Performance

Extract and analyze the gas transfer control behavior.

In [36]:
# Get batch data
batch = sim.data.batches[0]
t = np.array(batch.t_points)
DO = np.array(batch.y_points[6])
N = np.array(batch.N_list)
Fg = np.array(batch.Fg_list)
y_O2 = np.array(batch.y_O2_list)

# Calculate statistics
print("Gas Transfer Control Statistics:")
print(f"\nDissolved Oxygen:")
print(f"  Mean: {DO.mean():.4f} mmol/L")
print(f"  Std Dev: {DO.std():.4f} mmol/L")
print(f"  Min: {DO.min():.4f} mmol/L")
print(f"  Max: {DO.max():.4f} mmol/L")

print(f"\nStirring Speed:")
print(f"  Initial: {N[0]:.1f} rpm")
print(f"  Final: {N[-1]:.1f} rpm")
print(f"  Maximum: {N.max():.1f} rpm")

print(f"\nGas Flow Rate:")
print(f"  Initial: {Fg[0]:.4f} vvm")
print(f"  Final: {Fg[-1]:.4f} vvm")
print(f"  Maximum: {Fg.max():.4f} vvm")

print(f"\nO2 Mole Fraction:")
print(f"  Mean: {y_O2.mean():.3f}")
print(f"  Range: [{y_O2.min():.3f}, {y_O2.max():.3f}]")

Gas Transfer Control Statistics:

Dissolved Oxygen:
  Mean: 0.0100 mmol/L
  Std Dev: 0.0000 mmol/L
  Min: 0.0100 mmol/L
  Max: 0.0100 mmol/L

Stirring Speed:
  Initial: 30.0 rpm
  Final: 30.0 rpm
  Maximum: 30.0 rpm

Gas Flow Rate:
  Initial: 0.3000 vvm
  Final: 0.3000 vvm
  Maximum: 0.3000 vvm

O2 Mole Fraction:
  Mean: 0.210
  Range: [0.210, 0.210]


## 5. Multi-Batch Analysis

Run multiple batches and analyze gas consumption patterns.

In [37]:
# Run 5 batches
results_multi = sim.run_simulation(number_of_batches=5)

print(f"\nCompleted {len(sim.data.batches)} batches")
for i, batch in enumerate(sim.data.batches, 1):
    print(f"Batch {i}: {batch.t_end:.1f} h")

run_simulation execution time: 92.656 ms

Completed 5 batches
Batch 1: 100.0 h
Batch 2: 100.0 h
Batch 3: 100.0 h
Batch 4: 100.0 h
Batch 5: 100.0 h


### 5.1. Compare Gas Transfer Across Batches

In [38]:
# Plot gas transfer for batch 1
fig_gas_1 = plot_batch_gas_transfer(sim, batch_number=1)
fig_gas_1.update_layout(title_text="Batch 1: Gas Transfer Parameters")
fig_gas_1.show()

In [39]:
# Plot gas transfer for batch 3
fig_gas_3 = plot_batch_gas_transfer(sim, batch_number=3)
fig_gas_3.update_layout(title_text="Batch 3: Gas Transfer Parameters")
fig_gas_3.show()

In [40]:
# Plot gas transfer for batch 5
fig_gas_5 = plot_batch_gas_transfer(sim, batch_number=5)
fig_gas_5.update_layout(title_text="Batch 5: Gas Transfer Parameters")
fig_gas_5.show()

### 5.2. Plot Cumulative Gas Use

Visualize total gas consumption across all batches.

In [41]:
# Plot cumulative gas use for all 5 batches
fig_cumulative = plot_simulation_cumulative_gas_use(sim, max_batches=5)
fig_cumulative.show()

# Alternative: using the data object method
# fig_cumulative = sim.data.plot_cumulative_gas_use(max_batches=5)
# fig_cumulative.show()

### Interpretation:

**Cumulative Gas Use:**
- Shows total gas volume consumed over time
- Units: m³ (cubic meters) for readability
- Includes all batches in continuous timeline
- Slope indicates gas consumption rate
  - Steeper = higher oxygen demand (more cells, higher growth)
  - Flatter = lower oxygen demand (stationary phase)

**Cost Implications:**
- Higher cumulative gas use → higher operating costs
- Optimization target: minimize gas use while maintaining productivity
- Trade-off: O2 enrichment is expensive but may be necessary at high cell densities

## 6. Test with Different Operating Modes

Compare gas transfer requirements across operating modes.

### 6.1. Batch Mode

In [42]:
# Create batch mode simulation
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI"])

sim_batch = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    feed_strategy=None,
    perfusion_strategy=None,
    operating_mode=OperatingMode.BATCH
)

results_batch = sim_batch.run_simulation(number_of_batches=1)
batch1 = sim_batch.data.batches[0]
print(f"Batch mode completed in {batch1.t_end:.1f} hours")

# Plot
fig_gas_batch = plot_batch_gas_transfer(sim_batch, batch_number=1)
fig_gas_batch.show()

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/4] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/4] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
Successfully loaded parameters from 4 file(s).
run_simulation execution time: 12.413 ms
Batch mode completed in 94.1 hours


### 6.2. Perfusion Mode

In [46]:
# Create perfusion mode simulation
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "excess_kinetics_batch","fed_batch", "perfusion"])

sim_perfusion = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.EXCESS,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    perfusion_strategy=PerfusionStrategy.CSPR,
    operating_mode=OperatingMode.CONCENTRATED_FEDBATCH
)

results_perfusion = sim_perfusion.run_simulation(number_of_batches=1)
batch_data = sim_perfusion.data.batches[0]
print(f"Perfusion mode completed in {batch_data.t_end:.1f} hours")

fig_biomass_metabolites = batch_data.plot_biomass_metabolites()
fig_biomass_metabolites.show()
# Plot
fig_gas_perfusion = plot_batch_gas_transfer(sim_perfusion, batch_number=1)
fig_gas_perfusion.show()

Loading parameters from 7 file(s) sequentially...
  [1/7] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/7] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/7] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/7] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/7] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\excess_kinetics\excess_kinetics_batch.yaml
  [6/7] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakineti

## 7. Custom Layout Options

Demonstrate plot customization.

In [47]:
# Custom layout for presentation
custom_layout = {
    'height': 500,
    'width': 2000,
    'font': dict(size=14, family='Arial'),
    'title_font': dict(size=18),
    'template': 'plotly_white'
}

fig_custom = plot_batch_gas_transfer(
    sim,
    batch_number=1,
    layout_kwargs=custom_layout
)
fig_custom.show()

## Summary

This notebook demonstrated gas transfer visualization functions:

### Functions Tested:
1. **`plot_batch_gas_transfer()`** - Multi-panel gas transfer parameters
   - Dissolved oxygen concentration (DO)
   - Stirring speed (rpm)
   - Gas flow rate (vvm)
   - Oxygen mole fraction in gas phase

2. **`plot_simulation_cumulative_gas_use()`** - Cumulative gas consumption
   - Total gas volume used across batches
   - Useful for cost analysis and process economics

### Key Features:
- **PID Control Visualization**: Shows how controller maintains DO setpoint
- **Multi-variable Control**: Displays hierarchy (stirring → flow rate → O2 enrichment)
- **Batch Comparison**: Identify control performance variations
- **Cost Analysis**: Track cumulative gas use for economic evaluation
- **Interactive**: Plotly charts with hover details

### Use Cases:
- **Process Control Tuning**: Evaluate PID controller performance
- **Scale-up Studies**: Understand gas transfer limitations
- **Cost Optimization**: Minimize gas consumption while maintaining productivity
- **Troubleshooting**: Identify oxygen transfer bottlenecks
- **Operating Mode Comparison**: Compare gas requirements across strategies